# Eksperimen tambahan untuk JED (versi X8: atrisi)
**PENTING:** tambahkan dataset IFLS mentah (.dta, folder hh07_b2, hh07_bk, pce, hh14_trk, dst.) yang sama dengan notebook utama sebagai Input.

# Eksperimen tambahan untuk JED — urutan waktu bersih (X7) dan atrisi (X8)
Input: `umkm_ifls_v2.csv`, `umkm_ifls3_placebo.csv`, dan (untuk X8) dataset IFLS mentah (.dta) yang sama dengan notebook utama.

| Eksperimen | Pertanyaan | Output |
|---|---|---|
| X7a | Efek pinjaman **2007** pada penutupan usaha 2014, kovariat **2000** saja (urutan waktu bersih, N≈2.213) | `X7_timing.csv` |
| X7b | Efek pinjaman 2007, sampel penuh, kovariat 2007 non-pinjaman (kontemporer) | `X7_timing.csv` |
| X8 | Atrisi 2007→2014: siapa yang hilang, dan apakah estimasi utama berubah dengan bobot atrisi IPW | `X8_attrition_*.csv` |

CPU, `MODE="FULL"` → Save & Run All (±45–60 menit). X8 dilewati otomatis bila file .dta IFLS tidak ada di Input.

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_v] = "1"
import time, json, math, glob, re, warnings
import numpy as np, pandas as pd
from scipy import stats
from joblib import Parallel, delayed
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
warnings.filterwarnings("ignore")
try:
    import lightgbm as lgb; HAS_LGB = True
except Exception:
    HAS_LGB = False

MODE    = "FULL"
S_SPLIT = 20 if MODE == "FULL" else 2
S_ATTR  = 10 if MODE == "FULL" else 2
N_FOLDS = 5
SEED    = 2026
PS_CLIP = 0.01
N_JOBS  = -1
IFLS_ROOT = "/kaggle/input"
RUN_X7 = True    # X7: timing of credit; X8 runs automatically when the raw IFLS .dta files are attached
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./x2_out")
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:6.1f} mnt]", *a, flush=True)
log("MODE", MODE, "| lightgbm", HAS_LGB, "| CPU", os.cpu_count())

In [ ]:
def make_clf(kind="gbm", fast=False):
    it = 100 if fast else 300
    if kind == "rf":
        return RandomForestClassifier(n_estimators=200 if not fast else 100, min_samples_leaf=10, n_jobs=1, random_state=0)
    if HAS_LGB:
        return lgb.LGBMClassifier(n_estimators=it, learning_rate=0.1 if fast else 0.05, num_leaves=15,
                                  min_child_samples=20, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=1)
    return HistGradientBoostingClassifier(max_iter=it, learning_rate=0.1 if fast else 0.05,
                                          max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0,
                                          early_stopping=False)

def make_reg(kind="gbm", fast=False):
    it = 100 if fast else 300
    if kind == "rf":
        return RandomForestRegressor(n_estimators=200 if not fast else 100, min_samples_leaf=10, n_jobs=1, random_state=0)
    if HAS_LGB:
        return lgb.LGBMRegressor(n_estimators=it, learning_rate=0.1 if fast else 0.05, num_leaves=15,
                                 min_child_samples=20, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=1)
    return HistGradientBoostingRegressor(max_iter=it, learning_rate=0.1 if fast else 0.05,
                                         max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0,
                                         early_stopping=False)

def make_folds(scheme, groups, region_xy, k=5, seed=0, blocks_per_fold=2):
    # groups: label wilayah per observasi; region_xy: DataFrame index=label, kolom [x,y]
    n = len(groups); rng = np.random.default_rng(seed)
    if scheme == "random":
        return list(KFold(k, shuffle=True, random_state=seed).split(np.arange(n)))
    ug = np.asarray(region_xy.index)
    if scheme == "group":
        perm = rng.permutation(len(ug)); fold_of = dict(zip(ug[perm], np.arange(len(ug)) % k))
    elif scheme == "spatial":
        nb = min(len(ug), k * blocks_per_fold)
        lab = KMeans(nb, n_init=5, random_state=seed).fit_predict(region_xy[["x", "y"]].values)
        bperm = rng.permutation(nb); fold_of_block = {b: i % k for i, b in enumerate(bperm)}
        fold_of = {g: fold_of_block[l] for g, l in zip(ug, lab)}
    else:
        raise ValueError(scheme)
    f = np.array([fold_of[g] for g in groups])
    return [(np.where(f != j)[0], np.where(f == j)[0]) for j in range(k) if (f == j).any()]

def cluster_se(psi_centered, groups, w=None):
    # SE rata-rata (berbobot) dari skor terpusat, klaster pada groups
    w = np.ones_like(psi_centered) if w is None else w
    s = pd.Series(w * psi_centered).groupby(np.asarray(groups)).sum().values
    G = len(s)
    return math.sqrt(G / (G - 1) * np.sum(s ** 2)) / np.sum(w)

def _fit_pred(model, Xtr, ytr, Xte, binary):
    if binary:
        if ytr.min() == ytr.max():
            return np.full(len(Xte), float(ytr.mean()))
        return model.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    return model.fit(Xtr, ytr).predict(Xte)

def dml_aipw(Y, D, X, folds, binary_y, groups, w=None, clip=PS_CLIP, kind="gbm", fast=False):
    n = len(Y); e = np.zeros(n); m1 = np.zeros(n); m0 = np.zeros(n)
    for tr, te in folds:
        e[te] = _fit_pred(make_clf(kind, fast), X[tr], D[tr], X[te], True)
        for d, arr in ((1, m1), (0, m0)):
            sel = tr[D[tr] == d]
            mdl = make_clf(kind, fast) if binary_y else make_reg(kind, fast)
            arr[te] = _fit_pred(mdl, X[sel], Y[sel], X[te], binary_y)
    extreme = float(np.mean((e < clip) | (e > 1 - clip)))
    ec = np.clip(e, clip, 1 - clip)
    psi = m1 - m0 + D * (Y - m1) / ec - (1 - D) * (Y - m0) / (1 - ec)
    w_ = np.ones(n) if w is None else w
    ate = np.sum(w_ * psi) / np.sum(w_)
    se_iid = math.sqrt(np.sum(w_ ** 2 * (psi - ate) ** 2)) / np.sum(w_)
    se_cl = cluster_se(psi - ate, groups, w_)
    # ATT (AIPW)
    p = np.sum(w_ * D) / np.sum(w_)
    num = D * (Y - m0) - (1 - D) * ec / (1 - ec) * (Y - m0)
    att = np.sum(w_ * num) / np.sum(w_ * D)
    att_se = cluster_se((num - D * att) / p, groups, w_)
    return dict(ate=ate, se_iid=se_iid, se_cl=se_cl, att=att, att_se=att_se,
                extreme=extreme, psi=psi, e=ec, m1=m1, m0=m0)

def dr_learner(psi, X, folds, kind="gbm", fast=False):
    tau = np.zeros(len(psi))
    lo, hi = np.quantile(psi, [0.005, 0.995]); y = np.clip(psi, lo, hi)  # winsor ringan utk stabilitas
    for tr, te in folds:
        tau[te] = make_reg(kind, fast).fit(X[tr], y[tr]).predict(X[te])
    return tau

def ols_cluster(y, Xmat, groups, w=None):
    # WLS + SE klaster (CR1). Xmat sudah memuat konstanta.
    w = np.ones(len(y)) if w is None else w
    sw = np.sqrt(w); Xw = Xmat * sw[:, None]; yw = y * sw
    XtX_inv = np.linalg.pinv(Xw.T @ Xw)
    b = XtX_inv @ Xw.T @ yw
    u = yw - Xw @ b
    meat = np.zeros((Xmat.shape[1],) * 2)
    for g in np.unique(groups):
        idx = groups == g; s = Xw[idx].T @ u[idx]; meat += np.outer(s, s)
    n, k = Xmat.shape; G = len(np.unique(groups))
    V = XtX_inv @ meat @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    return b, np.sqrt(np.clip(np.diag(V), 0, None)), G

def blp_test(psi, tau_hat, groups, w=None):
    # psi_i = a + b*(tau_hat_i - mean) ; b>0 & signifikan => heterogenitas terdeteksi
    t = tau_hat - np.average(tau_hat, weights=w)
    Xm = np.column_stack([np.ones_like(t), t])
    b, se, G = ols_cluster(psi, Xm, np.asarray(groups), w)
    pval = 2 * (1 - stats.t.cdf(abs(b[1] / se[1]), df=G - 1)) if se[1] > 0 else np.nan
    return dict(ate_blp=b[0], beta_het=b[1], se_het=se[1], p_het=pval)

def gates(psi, tau_hat, groups, q=5, w=None):
    cut = pd.qcut(tau_hat, q, labels=False, duplicates="drop")
    rows = []
    for g in np.unique(cut):
        i = cut == g; ww = None if w is None else w[i]
        m = np.average(psi[i], weights=ww)
        rows.append(dict(group=int(g) + 1, gate=m, se=cluster_se(psi[i] - m, np.asarray(groups)[i], ww), n=int(i.sum())))
    return pd.DataFrame(rows)

def mundlak(X, g):
    df = pd.DataFrame(X); s = df.groupby(g).transform("sum"); c = pd.Series(g).map(pd.Series(g).value_counts()).values
    loo = (s.values - X) / np.maximum(c - 1, 1)[:, None]
    return np.column_stack([loo, np.log(c)])


def find_csv(name):
    hits = []
    for r in ["/kaggle/input", os.getcwd()]:
        hits += glob.glob(os.path.join(r, "**", name), recursive=True)
    hits = [h for h in hits if "/x2_out/" not in h]
    if not hits: raise FileNotFoundError(f"{name} tidak ditemukan di Input")
    return sorted(hits, key=len)[0]

def key(s): return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
def median_agg(est, se):
    est, se = np.asarray(est), np.asarray(se); m = np.median(est)
    return m, float(np.median(np.sqrt(se ** 2 + (est - m) ** 2)))
def rxy(df):
    return df.groupby("kab_kode")[["lon", "lat"]].mean().rename(columns={"lon": "x", "lat": "y"})
def seed_of(s): return SEED + s * 7919 if s > 0 else SEED

def dml_D(df, X, y, D, binary, seed, w, scheme="random"):
    f = make_folds(scheme, df["kab_kode"].values, rxy(df), N_FOLDS, seed=seed)
    return dml_aipw(y, D.astype(int), X, f, binary, df["kab_kode"].values, w)

def repeated(df, X, y, D, binary, w, S, scheme="random"):
    res = Parallel(n_jobs=N_JOBS)(delayed(dml_D)(df, X, y, D, binary, seed_of(s), w, scheme) for s in range(S))
    m, se = median_agg([r["ate"] for r in res], [r["se_cl"] for r in res])
    ma, sea = median_agg([r["att"] for r in res], [r["att_se"] for r in res])
    e = np.mean([r["e"] for r in res], axis=0)
    return dict(ate=m, se=se, p=2 * (1 - stats.norm.cdf(abs(m / se))), att=ma, att_se=sea,
                share_e_extreme=float(np.mean((e < .05) | (e > .95))), S=S)

## Data

In [ ]:
dat = pd.read_csv(find_csv("umkm_ifls_v2.csv"))
pl = pd.read_csv(find_csv("umkm_ifls3_placebo.csv"), dtype={"hhid14": str})
EDU = {1: 0, 90: 0, 2: 6, 11: 6, 17: 6, 72: 6, 14: 6, 3: 9, 4: 9, 12: 9, 73: 9, 5: 12, 6: 12, 15: 12, 74: 12,
       60: 14, 13: 16, 61: 16, 62: 18, 63: 21, 10: 0}
if pl["pendidikan_kepala00"].max() > 25:
    pl["pendidikan_kepala00"] = pl["pendidikan_kepala00"].map(EDU)
    pl["pendidikan_kepala00"] = pl["pendidikan_kepala00"].fillna(pl["pendidikan_kepala00"].median())
COVS = ["ihs_laba07", "pekerja07", "ihs_aset07", "umur_usaha07", "n_usaha07", "usaha_di_rumah07",
        "pinjam07", "ihs_pinjam07", "log_pce07", "hhsize07", "kepala_perempuan", "usia_kepala",
        "pendidikan_kepala", "perkotaan07", "n_lembaga_kredit_desa07", "jarak_lembaga_min07"]
COVS = [c for c in COVS if c in dat] + [c for c in dat.columns if c.endswith("_miss")]
OUTS = {"usaha_tutup": True, "ihs_laba": False, "tenaga_kerja": False}
def featC(df, cols):
    base = [c for c in cols if not c.endswith("_miss")]
    return np.column_stack([df[cols].values.astype(float), df[["lon", "lat"]].values.astype(float),
                            mundlak(df[base].values.astype(float), df["kab_kode"].values)])
log(f"N utama = {len(dat)} | plasebo = {len(pl)}")

## X7. Pinjaman 2007 → penutupan usaha 2014
**X7a (urutan waktu bersih):** RT dengan usaha di 2000 dan 2007; perlakuan = pinjaman 12 bulan sebelum survei 2007;
kovariat = karakteristik 2000 + lokasi + rerata kab/kota. Outcome diukur tujuh tahun setelah perlakuan.
**X7b (sampel penuh):** perlakuan sama; kovariat 2007 tanpa variabel pinjaman (diukur bersamaan dengan perlakuan).

In [ ]:
c00 = [c for c in pl.columns if (c.endswith("00") or c.endswith("00_miss")) and c not in ("prov00", "kab00") and not c.startswith("hhid")]
ph = pl[pl["usaha_tutup_0007"] == 0].assign(_k=key(pl["hhid14"]))[["_k"] + c00].drop_duplicates("_k")
dh = dat.assign(_k=key(dat["hhid14"])).merge(ph, on="_k", how="inner").reset_index(drop=True)
if "pinjam07_miss" in dh: dh = dh[dh["pinjam07_miss"] == 0].reset_index(drop=True)
Da = dh["pinjam07"].values.astype(int); Xa = featC(dh, c00)
dfull = dat[dat.get("pinjam07_miss", pd.Series(0, index=dat.index)) == 0].reset_index(drop=True)
covb = [c for c in COVS if "pinjam" not in c]
Db = dfull["pinjam07"].values.astype(int); Xb = featC(dfull, covb)
log(f"X7a N={len(dh)}, pinjam07={Da.mean():.3f} | X7b N={len(dfull)}, pinjam07={Db.mean():.3f}")
rows = []
for lab, df_, X_, D_ in ([] if not RUN_X7 else [("X7a: 2000 covariates (clean timing)", dh, Xa, Da), ("X7b: 2007 non-borrowing covariates (full sample)", dfull, Xb, Db)]):
    for o, b in OUTS.items():
        y = df_[o].values.astype(float)
        for wl in ["weighted", "unweighted"]:
            w = df_["weight"].values.astype(float) if wl == "weighted" else np.ones(len(df_))
            r = repeated(df_, X_, y, D_, b, w, S_SPLIT)
            base = float(np.average(y[D_ == 0], weights=w[D_ == 0]))
            rows.append(dict(spec=lab, outcome=o, weights=wl, n=len(df_), treated_share=float(D_.mean()),
                             control_mean=base, **r))
            log(f"{lab[:3]} {o:13s} {wl:10s} ATE={r['ate']:+.4f} ({r['se']:.4f}) p={r['p']:.3f}")
if rows:
    X7 = pd.DataFrame(rows); X7.to_csv(os.path.join(OUT, "X7_timing.csv"), index=False)
    print(X7.round(4).to_string(index=False))

## X8. Atrisi 2007 → 2014
Basis = semua RT dengan usaha non-pertanian di IFLS4 (2007), dibangun ulang dari .dta dengan definisi sama dengan notebook utama.
Terlacak = masuk sampel analisis (`umkm_ifls_v2.csv`). Model atrisi: LightGBM cross-fitted, P(terlacak | X2007, provinsi).
Lalu estimasi utama diulang dengan bobot = bobot IFLS × 1/P̂(terlacak).

In [ ]:
ALL_DTA = glob.glob(f"{IFLS_ROOT}/**/*.dta", recursive=True)
log(f"{len(ALL_DTA)} file .dta ditemukan di Input")
_dirs = sorted(set(os.path.dirname(f) for f in ALL_DTA))
print("Folder berisi .dta:"); [print("  ", d, "->", len([f for f in ALL_DTA if os.path.dirname(f) == d]), "file") for d in _dirs]
print("Contoh nama file:", sorted(set(os.path.basename(f) for f in ALL_DTA))[:60])
_need = ["b2_nt1", "b2_nt2", "b2_bh", "bk_ar1", "bk_sc", "pce07nom"]
for n in _need:
    c = [f for f in ALL_DTA if os.path.basename(f).lower() == n + ".dta"]
    print(f"  {n:9s}: {len(c)} kandidat", [os.path.basename(os.path.dirname(x)) for x in c])
def find_dta(folder_hint, fname):
    cand = [f for f in ALL_DTA if os.path.basename(f).lower() == fname.lower() + ".dta"]
    hits = [f for f in cand if folder_hint in f] or [f for f in cand if folder_hint.split("_")[0] in f]
    if not hits:
        raise FileNotFoundError(f"{fname}.dta ({folder_hint}); kandidat: {cand[:5]}")
    return sorted(hits, key=len)[0]
def rd(folder_hint, fname, cols=None):
    df = pd.read_stata(find_dta(folder_hint, fname), convert_categoricals=False)
    df.columns = [c.lower() for c in df.columns]
    return df[[c for c in cols if c in df.columns]] if cols else df
def sid(s): return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
num = lambda s: pd.to_numeric(s, errors="coerce")

def build_base07():
    nt1 = rd("hh07_b2", "b2_nt1", ["hhid07", "nt01"]); nt1["hhid07"] = sid(nt1.hhid07)
    base = nt1.loc[num(nt1.nt01) == 1, ["hhid07"]].drop_duplicates()
    nt2 = rd("hh07_b2", "b2_nt2"); nt2["hhid07"] = sid(nt2.hhid07)
    op = nt2[num(nt2.get("nt18")).fillna(1) == 1].copy()
    for c in ["nt09", "nt22", "nt23", "nt10a", "nt10b", "nt10c1", "nt10c4", "nt10h", "nt15myr", "nt05b"]:
        op[c] = num(op[c]) if c in op else np.nan
    op["aset"] = op[["nt10a", "nt10b", "nt10c1", "nt10c4", "nt10h"]].sum(axis=1, min_count=1)
    op["pekerja"] = op[["nt22", "nt23"]].sum(axis=1, min_count=1)
    op["umur"] = 2007 - op["nt15myr"].where(op["nt15myr"].between(1900, 2007))
    g = op.groupby("hhid07").agg(laba07=("nt09", "sum"), pekerja07=("pekerja", "sum"), aset07=("aset", "sum"),
                                 umur_usaha07=("umur", "max"), n_usaha07=("nt09", "size"),
                                 usaha_di_rumah07=("nt05b", lambda s: float((s == 1).any()))).reset_index()
    base = base.merge(g, on="hhid07", how="left")
    bh = rd("hh07_b2", "b2_bh", ["hhid07", "bh07", "bh10"]); bh["hhid07"] = sid(bh.hhid07)
    bh["pinjam07"] = (num(bh.bh07) == 1).astype(float)
    bh["ihs_pinjam07"] = np.arcsinh(num(bh.bh10).fillna(0) * bh["pinjam07"])
    base = base.merge(bh[["hhid07", "pinjam07", "ihs_pinjam07"]].drop_duplicates("hhid07"), on="hhid07", how="left")
    pce = rd("pce", "pce07nom"); pce["hhid07"] = sid(pce.hhid07)
    pc = next((c for c in pce.columns if re.fullmatch(r"(pce|pcexp|pce07|hhexp_pc|pcexpend)", c)), None)
    pce["log_pce07"] = np.log(num(pce[pc]).where(num(pce[pc]) > 0)) if pc else np.nan
    pce["hhsize07"] = num(pce.get("hhsize"))
    base = base.merge(pce[["hhid07", "log_pce07", "hhsize07"]].drop_duplicates("hhid07"), on="hhid07", how="left")
    ar = rd("hh07_bk", "bk_ar1", ["hhid07", "ar02b", "ar07", "ar09", "ar16"]); ar["hhid07"] = sid(ar.hhid07)
    hd = ar[num(ar.ar02b) == 1].drop_duplicates("hhid07")
    hd = hd.assign(kepala_perempuan=(num(hd.ar07) == 3).astype(float),
                   usia_kepala=num(hd.ar09).where(num(hd.ar09).between(10, 100)),
                   pendidikan_kepala=num(hd.ar16).map(EDU))
    base = base.merge(hd[["hhid07", "kepala_perempuan", "usia_kepala", "pendidikan_kepala"]], on="hhid07", how="left")
    sc = rd("hh07_bk", "bk_sc"); sc["hhid07"] = sid(sc.hhid07)
    sc["perkotaan07"] = (num(sc.get("sc05")) == 1).astype(float)
    pcol = next((c for c in ["sc010707", "sc01_07_07", "sc01"] if c in sc.columns), None)
    sc["prov07"] = num(sc[pcol]) if pcol else np.nan
    base = base.merge(sc[["hhid07", "perkotaan07", "prov07"]].drop_duplicates("hhid07"), on="hhid07", how="left")
    for c, f in [("laba07", "ihs_laba07"), ("aset07", "ihs_aset07")]:
        base[f] = np.arcsinh(base[c].fillna(0))
    base.loc[base.pekerja07 >= 999, "pekerja07"] = np.nan
    return base

X8_rows, X8_bal = [], []
try:
    base = build_base07()
    tracked_ids = set(key(dat["hhid07"].astype(str)))
    base["tracked"] = key(base["hhid07"]).isin(tracked_ids).astype(int)
    log(f"Basis 2007: {len(base)} RT usaha | terlacak ke sampel analisis: {base.tracked.mean():.1%}")
    acov = ["ihs_laba07", "pekerja07", "ihs_aset07", "umur_usaha07", "n_usaha07", "usaha_di_rumah07", "pinjam07",
            "ihs_pinjam07", "log_pce07", "hhsize07", "kepala_perempuan", "usia_kepala", "pendidikan_kepala", "perkotaan07"]
    for c in acov:
        x = base[c].astype(float); m1 = x[base.tracked == 1].mean(); m0 = x[base.tracked == 0].mean()
        sd = math.sqrt((x[base.tracked == 1].var() + x[base.tracked == 0].var()) / 2) or np.nan
        X8_bal.append(dict(variable=c, tracked=m1, attrited=m0, smd=(m1 - m0) / sd, share_missing=float(x.isna().mean())))
    XA = base[acov].copy()
    for c in acov:
        XA[c + "_miss"] = XA[c].isna().astype(float); XA[c] = XA[c].fillna(XA[c].median())
    if base["prov07"].notna().any():
        XA = pd.concat([XA, pd.get_dummies(base["prov07"].fillna(-1).astype(int), prefix="p").astype(float)], axis=1)
    XA = XA.values.astype(float); ya = base["tracked"].values
    p_hat = np.zeros(len(base))
    for tr, te in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(XA):
        p_hat[te] = _fit_pred(make_clf(), XA[tr], ya[tr], XA[te], True)
    auc = roc_auc_score(ya, p_hat)
    log(f"AUC model atrisi = {auc:.3f}")
    base["p_track"] = np.clip(p_hat, 0.05, 1.0)
    pmap = dict(zip(key(base["hhid07"]), base["p_track"]))
    p_dat = key(dat["hhid07"].astype(str)).map(pmap).fillna(base["p_track"][base.tracked == 1].median()).values
    w_attr = dat["weight"].values / p_dat; w_attr = w_attr / w_attr.mean() * dat["weight"].mean()
    X8_rows.append(dict(outcome="(model)", weights="attrition model", ate=np.nan, se=np.nan, p=np.nan,
                        n=len(base), note=f"tracked={base.tracked.mean():.3f}; AUC={auc:.3f}; "
                        f"IPW weight max/median={np.max(1/p_dat)/np.median(1/p_dat):.2f}"))
    XC = featC(dat, COVS); D0 = dat["kredit"].values.astype(int)
    for o, b in OUTS.items():
        y = dat[o].values.astype(float)
        for wl, w in [("IFLS weights", dat["weight"].values.astype(float)), ("IFLS weights x attrition IPW", w_attr)]:
            r = repeated(dat, XC, y, D0, b, w, S_ATTR)
            X8_rows.append(dict(outcome=o, weights=wl, ate=r["ate"], se=r["se"], p=r["p"], n=len(dat), note=f"S={S_ATTR}"))
            log(f"X8 {o:13s} {wl:28s} ATE={r['ate']:+.4f} ({r['se']:.4f})")
except FileNotFoundError as e:
    log("X8 dilewati: file IFLS .dta tidak ditemukan di Input ->", e)
except Exception as e:
    log("X8 gagal:", repr(e)[:300])
pd.DataFrame(X8_rows).to_csv(os.path.join(OUT, "X8_attrition_estimates.csv"), index=False)
pd.DataFrame(X8_bal).to_csv(os.path.join(OUT, "X8_attrition_balance.csv"), index=False)
if X8_rows: print(pd.DataFrame(X8_rows).round(4).to_string(index=False))
if X8_bal: print(pd.DataFrame(X8_bal).round(3).to_string(index=False))

## Ringkasan & ZIP

In [ ]:
lines = [f"# Eksperimen tambahan JED (MODE={MODE}, S={S_SPLIT}, lightgbm={HAS_LGB})", ""]
for nm, fn in [("X7 timing", "X7_timing.csv"), ("X8 attrition estimates", "X8_attrition_estimates.csv"), ("X8 attrition balance", "X8_attrition_balance.csv")]:
    p = os.path.join(OUT, fn)
    if os.path.exists(p) and os.path.getsize(p) > 5:
        lines += [f"## {nm}", "", "```", pd.read_csv(p).round(4).to_string(index=False), "```", ""]
open(os.path.join(OUT, "X2_ringkasan.md"), "w").write("\n".join(lines))
import shutil
zp = shutil.make_archive(os.path.join("/tmp", f"X2_results_{MODE}"), "zip", OUT)
shutil.move(zp, os.path.join(OUT, os.path.basename(zp)))
log("Selesai ->", os.path.join(OUT, os.path.basename(zp)))